# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an end-to-end template for loading and exploring the [FAIR² Mental Health Survey Dataset](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# For visualization later
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema catalog URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n\nPublished: {metadata.datePublished} (version {metadata.version})\nLicense: {metadata.license}\nCite as: {getattr(metadata, 'citeAs', '<not provided>')}")

## 2. Data Overview
Review available record sets, their fields and column `@id`s.

In [ ]:
# List all available Record Sets and Fields defined by their @id
print("Available record sets in this dataset (by @id and name):\n")
# The mlcroissant high-level API exposes dataset.record_set_ids and dataset.record_set_details
record_set_ids = dataset.record_set_ids
for rs_id in record_set_ids:
    details = dataset.record_set_details(rs_id)
    print(f"  @id: {rs_id}  Name: {details['name']}")

# View example fields and columns for the primary record set (using @id)
# We'll assume the first record set is the main one (usually the case for survey datasets)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    details = dataset.record_set_details(main_record_set_id)
    print(f"\nFields and columns for Record Set {main_record_set_id} ({details['name']}):\n")
    for field in details['fields']:
        print(f"  Field @id: {field['@id']}, Column @id: {field['column']}, Name: {field['name']}, DataType: {field['dataType']}")
else:
    print("No record sets found in the schema.")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis using their `@id`. Use the field and column `@id`s from the overview above.

In [ ]:
# Extract data from each record set and store as Pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for Record Set @id {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns in DataFrame: {df.columns.tolist()}")
    print(df.head(2), "\n")

# Use the main record set for further analysis
df_main = dataframes[main_record_set_id]
print(f"\nLoaded DataFrame shape: {df_main.shape}")
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform basic EDA: filtering, normalization, and grouping using fields referenced by their `@id` (column names).

For example, we can:
- Select a numeric field such as the GAD-7 total score or PHQ-9 total score (`gad7_total_score` or `phq9_total_score`)
- Filter rows above a threshold
- Normalize this score
- Group by a demographic field (e.g., `level_of_education` or `gender`)

_**(Adjust the field/column @id as needed based on the record set description above. Here we illustrate with likely field names, please revise to exact @id as appropriate.)_

In [ ]:
# Choose field @id for total GAD-7 score (adjust to actual @id from overview, e.g. 'gad7_total_score')
numeric_field = None
for col in df_main.columns:
    if 'gad7' in col.lower() and ('score' in col.lower() or 'total' in col.lower()):
        numeric_field = col
        break
if numeric_field is None:
    # Fallback to PHQ-9 score
    for col in df_main.columns:
        if 'phq9' in col.lower() and ('score' in col.lower() or 'total' in col.lower()):
            numeric_field = col
            break
if numeric_field is None:
    raise Exception("Didn't find a suitable numeric score field. Please check the DataFrame column names above.")
print(f"Using numeric field (by @id): {numeric_field}")

# Filter for high scores (e.g., GAD-7 > 10) -- indicative of moderate/severe anxiety
threshold = 10
filtered_df = df_main[df_main[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {filtered_df.shape[0]} found.")
print(filtered_df[[numeric_field]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping by 'level_of_education' if available (@id may differ)
group_field = None
for col in df_main.columns:
    if 'educat' in col.lower():
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMean {numeric_field} by {group_field} (for filtered cases):\n", grouped_df)
else:
    print("No 'education' group field found.")

## 5. Visualization
Let's visualize the distribution of our chosen numeric scale, and compare means by a demographic variable.

If available, we'll plot:
- The histogram of the selected score across all participants
- Boxplots of the score by gender or education

In [ ]:
# Histogram for the main numeric score (e.g., GAD-7)
plt.figure(figsize=(7, 4))
sns.histplot(df_main[numeric_field], bins=15, kde=True, color='tab:blue')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.title(f'Distribution of {numeric_field}')
plt.tight_layout()
plt.show()

# Boxplot by group (gender or education), if one exists
candidate_groups = [col for col in df_main.columns if ("gender" in col.lower() or "educat" in col.lower())]
if candidate_groups:
    group = candidate_groups[0]
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=group, y=numeric_field, data=df_main)
    plt.title(f"{numeric_field} by {group}")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print("No obvious categorical demographic group (like gender or education) found for boxplot.")

## 6. Conclusion
This notebook demonstrated how to:
- Load metadata and data from a FAIR² Croissant-formatted dataset using `mlcroissant`
- Review available record sets and fields by their `@id`
- Extract and examine survey data
- Process and analyze numeric fields
- Visualize key patterns (distribution, group differences)

Further analysis can include more advanced modeling, missing value imputation, or deeper subgroup explorations.

**Remember:** All references to fields, columns, and record sets in data extraction and EDA have used their `@id` as shown in the dataset schema.